In [9]:
import os
import numpy as np
import joblib
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [10]:
PROCESSED_DIR = "data/processed"
MODELS_DIR    = "models"

In [ ]:
def load_splits():
    def load(name):
        X = np.load(os.path.join(PROCESSED_DIR, f"X_{name}.npy"))
        y = np.load(os.path.join(PROCESSED_DIR, f"y_{name}.npy"))
        return X, y

    train = load("train")
    val   = load("val")
    test  = load("test")
    y_raw = np.load(os.path.join(PROCESSED_DIR, "test_y_raw.npy"))
    scaler_y = joblib.load(os.path.join(MODELS_DIR, "scaler_y.pkl"))

    return train, val, test, y_raw, scaler_y

In [12]:
def flatten(X):
    """
    Ridge/XGBoost require 2D input.
    (n_days, 96, 7) → (n_days, 672)
    Bi-LSTM keeps the 3D shape — that flattening happens in its own train.py.
    """
    return X.reshape(len(X), -1)

In [13]:
def train_ridge(X_train, y_train):
    print("[1/3] Training RidgeCV ...")
    print(f"  Input shape: {X_train.shape}  →  Output shape: {y_train.shape}")

    model = RidgeCV(
        alphas=[0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
        cv=5,
        scoring="neg_mean_absolute_error"
    )
    model.fit(X_train, y_train)
    print(f"  Best alpha (CV): {model.alpha_}")
    return model

In [14]:
def evaluate(model, X_test_flat, y_test_raw, scaler_y, split_name="Test"):
    print(f"\n[2/3] Evaluating on {split_name} set ...")

    y_pred_sc = model.predict(X_test_flat)                        # scaled predictions
    y_pred    = scaler_y.inverse_transform(y_pred_sc)             # back to original scale

    mae   = mean_absolute_error(y_test_raw, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_test_raw, y_pred))
    smape = np.mean(
        2 * np.abs(y_test_raw - y_pred) /
        (np.abs(y_test_raw) + np.abs(y_pred) + 1e-8)
    ) * 100
    mase_denom = np.mean(np.abs(np.diff(y_test_raw, axis=0)))
    mase  = mae / mase_denom if mase_denom > 0 else float("nan")

    print(f"  {'─'*35}")
    print(f"  MAE:   {mae:.4f}")
    print(f"  RMSE:  {rmse:.4f}")
    print(f"  sMAPE: {smape:.2f}%")
    print(f"  MASE:  {mase:.4f}  (< 1.0 beats naive baseline)")
    print(f"  {'─'*35}")
    return y_pred

In [15]:
def save_model(model):
    print("\n[3/3] Saving model ...")
    os.makedirs(MODELS_DIR, exist_ok=True)
    path = os.path.join(MODELS_DIR, "ridge.pkl")
    joblib.dump(model, path)
    print(f"  Saved → {path}")

In [17]:
if __name__ == "__main__":

    # Load
    (X_train, y_train), (X_val, y_val), (X_test, y_test), y_test_raw, scaler_y = load_splits()

    # Flatten: (n, 96, 7) → (n, 672)
    X_train_flat = flatten(X_train)
    X_val_flat   = flatten(X_val)
    X_test_flat  = flatten(X_test)

    print(f"Train: {X_train_flat.shape} | Val: {X_val_flat.shape} | Test: {X_test_flat.shape}")

    # Train
    model = train_ridge(X_train_flat, y_train)

    # Optional: also check performance on validation set
    evaluate(model, X_val_flat,
             scaler_y.inverse_transform(y_val), scaler_y, split_name="Validation")

    # Final evaluation on test set (unscaled)
    y_pred = evaluate(model, X_test_flat, y_test_raw, scaler_y, split_name="Test")

    # Compare against naive baseline
    # result = naive_last_week(y_test_raw)
    # if result is not None:
    #     _, mae_naive = result
    #     mae_model = mean_absolute_error(y_test_raw, y_pred)
    #     improvement = (mae_naive - mae_model) / mae_naive * 100
    #     print(f"  Ridge improvement over naive: {improvement:+.1f}%")

    # Save
    save_model(model)

    print("\n  Done. Next: python src/train.py (xgboost) or python src/evaluate.py")

Train: (360, 1248) | Val: (77, 1248) | Test: (78, 1248)
[1/3] Training RidgeCV ...
  Input shape: (360, 1248)  →  Output shape: (360, 96)
  Best alpha (CV): 0.1

[2/3] Evaluating on Validation set ...
  ───────────────────────────────────
  MAE:   15.2605
  RMSE:  36.5489
  sMAPE: 143.69%
  MASE:  0.1000  (< 1.0 beats naive baseline)
  ───────────────────────────────────

[2/3] Evaluating on Test set ...
  ───────────────────────────────────
  MAE:   22.6101
  RMSE:  52.8956
  sMAPE: 116.87%
  MASE:  0.0686  (< 1.0 beats naive baseline)
  ───────────────────────────────────

[3/3] Saving model ...
  Saved → models/ridge.pkl

  Done. Next: python src/train.py (xgboost) or python src/evaluate.py


/opt/miniconda3/envs/aiengine/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:267: UserWarning: Singular matrix in solving dual problem. Using least-squares solution instead.
  warnings.warn(
/opt/miniconda3/envs/aiengine/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 7.712670779813635e-11.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/aiengine/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 5.107660072667386e-09.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/aiengine/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 5.149936033177482e-10.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/opt/miniconda3/envs/aien